In [5]:
!git clone https://github.com/OpenNeuroDatasets/ds007328.git

Cloning into 'ds007328'...
remote: Enumerating objects: 19359, done.
remote: Counting objects: 100% (19359/19359), done.
remote: Compressing objects: 100% (11006/11006), done.
remote: Total 19359 (delta 5340), reused 19357 (delta 5338), pack-reused 0 (from 0)
Receiving objects: 100% (19359/19359), 10.30 MiB | 19.82 MiB/s, done.
Resolving deltas: 100% (5340/5340), done.


In [2]:
import os
import hashlib
import time
from pathlib import Path
from typing import List


def sha256_file(filepath: Path) -> bytes:
    hasher = hashlib.sha256()
    try:
        with open(filepath, "rb") as f:
            for chunk in iter(lambda: f.read(8192), b""):
                hasher.update(chunk)
        return hasher.digest()
    except Exception as e:
        return None


def build_merkle_tree(hashes: List[bytes]) -> List[List[bytes]]:
    tree = [hashes]

    while len(tree[-1]) > 1:
        level = []
        nodes = tree[-1]

        for i in range(0, len(nodes), 2):
            left = nodes[i]
            right = nodes[i + 1] if i + 1 < len(nodes) else left
            combined = hashlib.sha256(left + right).digest()
            level.append(combined)

        tree.append(level)

    return tree


def get_merkle_proof(tree: List[List[bytes]], index: int) -> List[bytes]:
    proof = []
    for level in tree[:-1]:
        sibling_index = index ^ 1
        if sibling_index < len(level):
            proof.append(level[sibling_index])
        index //= 2
    return proof


def verify_proof(leaf: bytes, proof: List[bytes], root: bytes, index: int) -> bool:
    computed_hash = leaf

    for sibling in proof:
        if index % 2 == 0:
            computed_hash = hashlib.sha256(computed_hash + sibling).digest()
        else:
            computed_hash = hashlib.sha256(sibling + computed_hash).digest()
        index //= 2

    return computed_hash == root


def collect_files(dataset_path: str) -> List[Path]:
    files = []
    for root, _, filenames in os.walk(dataset_path):
        for name in filenames:
            filepath = Path(root) / name
            if filepath.is_file() and not filepath.is_symlink():
                files.append(filepath)
    return files


def run_benchmark(dataset_path: str):
    print("Collecting files...")
    files = collect_files(dataset_path)
    print(f"Total candidate files: {len(files)}")

    print("Hashing files...")
    start_hash = time.time()

    hashes = []
    skipped = 0

    for f in files:
        h = sha256_file(f)
        if h:
            hashes.append(h)
        else:
            skipped += 1

    end_hash = time.time()

    print(f"Valid hashed files: {len(hashes)}")
    print(f"Skipped files: {skipped}")
    print(f"Hashing time: {(end_hash - start_hash):.4f} s")

    print("Building Merkle Tree...")
    start_build = time.time()
    tree = build_merkle_tree(hashes)
    end_build = time.time()

    root = tree[-1][0]
    print(f"Merkle root: {root.hex()}")
    print(f"Build time: {(end_build - start_build):.4f} s")

    print("Generating proof...")
    start_proof = time.time()
    proof = get_merkle_proof(tree, 0)
    end_proof = time.time()

    print(f"Proof generation time: {(end_proof - start_proof)*1000:.4f} ms")

    print("Verifying proof...")
    start_verify = time.time()
    result = verify_proof(hashes[0], proof, root, 0)
    end_verify = time.time()

    print(f"Verification result: {result}")
    print(f"Verification time: {(end_verify - start_verify)*1000:.4f} ms")

    print("\n----- Summary -----")
    print(f"Files hashed: {len(hashes)}")
    print(f"Hashing time: {(end_hash - start_hash):.4f} s")
    print(f"Merkle build time: {(end_build - start_build):.4f} s")
    print(f"Verification time: {(end_verify - start_verify)*1000:.4f} ms")


if __name__ == "__main__":
    DATASET_PATH = "ds007328"
    run_benchmark(DATASET_PATH)

Total candidate files: 3151
Hashing files...
Valid hashed files: 3151
Skipped files: 0
Hashing time: 0.7281 s
Building Merkle Tree...
Merkle root: 657088d920800fc93a822c5f94e1404a1eda234c4e880aac2e6bb4998c05393c
Build time: 0.0073 s
Generating proof...
Proof generation time: 0.0131 ms
Verifying proof...
Verification result: True
Verification time: 0.0486 ms

----- Summary -----
Files hashed: 3151
Hashing time: 0.7281 s
Merkle build time: 0.0073 s
Verification time: 0.0486 ms


In [3]:
!pip install pycryptodome

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 68.9 MB/s eta 0:00:00


In [4]:
import hashlib
import os
import time
from pathlib import Path
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import secrets

# -------------------------------
# Utility: SHA-256 Hashing
# -------------------------------
def sha256_file(filepath: Path) -> bytes:
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hasher.update(chunk)
    return hasher.digest()

# -------------------------------
# Merkle Tree Construction
# -------------------------------
class MerkleTree:
    def __init__(self, leaves: list):
        self.leaves = leaves
        self.levels = []
        self.build_tree()

    def build_tree(self):
        current_level = self.leaves
        self.levels.append(current_level)
        while len(current_level) > 1:
            next_level = []
            for i in range(0, len(current_level), 2):
                left = current_level[i]
                right = current_level[i+1] if i+1 < len(current_level) else left
                next_level.append(hashlib.sha256(left + right).digest())
            current_level = next_level
            self.levels.append(current_level)

    def root(self):
        return self.levels[-1][0] if self.levels else None

# -------------------------------
# AES-256 Encryption/Decryption
# -------------------------------
def encrypt_file(data: bytes, key: bytes) -> bytes:
    cipher = AES.new(key, AES.MODE_GCM)
    ciphertext, tag = cipher.encrypt_and_digest(data)
    return cipher.nonce + tag + ciphertext

def decrypt_file(enc_data: bytes, key: bytes) -> bytes:
    nonce = enc_data[:16]
    tag = enc_data[16:32]
    ciphertext = enc_data[32:]
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    return cipher.decrypt_and_verify(ciphertext, tag)

# -------------------------------
# Ephemeral Key Derivation
# -------------------------------
def derive_ephemeral_key(master_key: bytes, user_pubkey: bytes, nonce: bytes, expiry: float) -> bytes:
    # simple derivation: hash(master + user_pub + nonce + expiry)
    data = master_key + user_pubkey + nonce + expiry.to_bytes(8, 'big')
    return hashlib.sha256(data).digest()

# -------------------------------
# End-to-End Benchmark
# -------------------------------
def run_benchmark(dataset_path: str, subset_size: int = 20):
    path = Path(dataset_path)
    files = [f for f in path.rglob("*") if f.is_file()]
    files = files[:subset_size]  # small subset for demo
    print(f"Benchmarking {len(files)} files...")

    # Hashing
    start = time.time()
    file_hashes = [sha256_file(f) for f in files]
    hash_time = time.time() - start
    print(f"Hashing time: {hash_time:.4f} s")

    # Build Merkle Tree
    start = time.time()
    mt = MerkleTree(file_hashes)
    merkle_time = time.time() - start
    print(f"Merkle root: {mt.root().hex()}")
    print(f"Merkle build time: {merkle_time:.4f} s")

    # Simulate ephemeral key issuance
    master_key = secrets.token_bytes(32)
    user_pub = secrets.token_bytes(32)
    nonce = secrets.token_bytes(16)
    expiry = int(time.time()) + 60  # 60 seconds validity
    start = time.time()
    e_key = derive_ephemeral_key(master_key, user_pub, nonce, expiry)
    key_time = time.time() - start
    print(f"Ephemeral key derivation time: {key_time*1000:.2f} ms")

    # AES encrypt/decrypt one file
    sample_file = files[0]
    data = sample_file.read_bytes()
    enc_data = encrypt_file(data, master_key)
    start = time.time()
    dec_data = decrypt_file(enc_data, master_key)
    dec_time = time.time() - start
    assert dec_data == data, "Decryption failed!"
    print(f"AES-256 decryption time for one file: {dec_time*1000:.2f} ms")

    # Summary
    print("\n----- Summary -----")
    print(f"Files hashed: {len(files)}")
    print(f"Hashing time: {hash_time:.4f} s")
    print(f"Merkle tree build time: {merkle_time:.4f} s")
    print(f"Ephemeral key derivation: {key_time*1000:.2f} ms")
    print(f"AES-256 decryption: {dec_time*1000:.2f} ms")

# -------------------------------
# Main
# -------------------------------
if __name__ == "__main__":
    DATASET_PATH = "ds007328"  # مسیر dataset محلی
    run_benchmark(DATASET_PATH)

Benchmarking 20 files...
Hashing time: 0.0019 s
Merkle root: f5ca029f1b429256a40812b288e91ed29fa97910fe1070a308b775fbfcbb4ef8
Merkle build time: 0.0000 s
Ephemeral key derivation time: 0.01 ms
AES-256 decryption time for one file: 0.33 ms

----- Summary -----
Files hashed: 20
Hashing time: 0.0019 s
Merkle tree build time: 0.0000 s
Ephemeral key derivation: 0.01 ms
AES-256 decryption: 0.33 ms


In [10]:
import hashlib
import time
import random
from pathlib import Path
import tracemalloc

# -------------------------
# Hashing
# -------------------------
def sha256_file(filepath):
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hasher.update(chunk)
    return hasher.digest()

# -------------------------
# Merkle Tree
# -------------------------
def build_merkle(hashes):
    level = hashes
    while len(level) > 1:
        next_level = []
        for i in range(0, len(level), 2):
            left = level[i]
            right = level[i+1] if i+1 < len(level) else left
            next_level.append(hashlib.sha256(left + right).digest())
        level = next_level
    return level[0]

# -------------------------
# Scalability Benchmark
# -------------------------
def scalability_test(dataset_path):
    files = [f for f in Path(dataset_path).rglob("*") if f.is_file()]
    sizes = [100, 500, 1000, 2000, len(files)]

    results = []

    for n in sizes:
        subset = files[:n]

        tracemalloc.start()
        start_hash = time.time()
        hashes = [sha256_file(f) for f in subset]
        hash_time = time.time() - start_hash

        start_build = time.time()
        root = build_merkle(hashes)
        build_time = time.time() - start_build

        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()

        results.append((n, hash_time, build_time, peak/1024/1024))

        print(f"Files: {n} | Hash: {hash_time:.4f}s | Build: {build_time:.4f}s | Peak RAM: {peak/1024/1024:.2f} MB")

    return results

if __name__ == "__main__":
    scalability_test("ds007328")

Files: 100 | Hash: 0.0171s | Build: 0.0010s | Peak RAM: 0.02 MB
Files: 500 | Hash: 0.0435s | Build: 0.0055s | Peak RAM: 0.06 MB
Files: 1000 | Hash: 0.0872s | Build: 0.0126s | Peak RAM: 0.12 MB
Files: 2000 | Hash: 0.2278s | Build: 0.0296s | Peak RAM: 0.25 MB
Files: 3151 | Hash: 0.3650s | Build: 0.0253s | Peak RAM: 0.39 MB


In [12]:
import hashlib
import secrets
import time
from multiprocessing import Pool

MASTER_KEY = secrets.token_bytes(32)

def derive_key(user_id):
    nonce = secrets.token_bytes(16)
    data = MASTER_KEY + user_id.to_bytes(4, "big") + nonce
    return hashlib.sha256(data).digest()

def simulate_concurrent(users):
    start = time.time()
    with Pool() as p:
        p.map(derive_key, range(users))
    total_time = time.time() - start
    print(f"{users} concurrent users -> {total_time:.4f}s total")

if __name__ == "__main__":
    for u in [10, 50, 100, 200]:
        simulate_concurrent(u)

10 concurrent users -> 0.0293s total
50 concurrent users -> 0.0277s total
100 concurrent users -> 0.0286s total
200 concurrent users -> 0.0329s total


In [13]:
import hashlib
import json

# Simulate watermark embedding
def embed_watermark(data, user_id):
    return data + f"::USER_{user_id}".encode()

def extract_watermark(data):
    marker = data.split(b"::USER_")
    if len(marker) > 1:
        return marker[-1].decode()
    return None

# Simulate distribution chain
def simulate_leakage():
    original = b"MEDICAL_DATA_SAMPLE"

    user_chain = [1, 2, 3]  # A -> B -> C
    data = original

    for u in user_chain:
        data = embed_watermark(data, u)

    leaked_copy = data

    offender = extract_watermark(leaked_copy)
    print(f"Leak detected from User ID: {offender}")

if __name__ == "__main__":
    simulate_leakage()

Leak detected from User ID: 3


In [16]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# -----------------------
# Your experimental data
# -----------------------

files = np.array([100, 500, 1000, 2000, 3151]).reshape(-1, 1)

hash_times = np.array([0.0171, 0.0435, 0.0872, 0.2278, 0.3650])
build_times = np.array([0.0010, 0.0055, 0.0126, 0.0296, 0.0253])

# -----------------------
# Linear Regression - Hash
# -----------------------

model_hash = LinearRegression()
model_hash.fit(files, hash_times)
pred_hash = model_hash.predict(files)

r2_hash = r2_score(hash_times, pred_hash)

# -----------------------
# Linear Regression - Merkle Build
# -----------------------

model_build = LinearRegression()
model_build.fit(files, build_times)
pred_build = model_build.predict(files)

r2_build = r2_score(build_times, pred_build)

print("Hashing Regression:")
print("Slope:", model_hash.coef_[0])
print("R²:", r2_hash)

print("\nMerkle Build Regression:")
print("Slope:", model_build.coef_[0])
print("R²:", r2_build)

Hashing Regression:
Slope: 0.00011795587223974284
R²: 0.9917486090461499

Merkle Build Regression:
Slope: 8.965933355382855e-06
R²: 0.799012181838201
